In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np


In [2]:
# Carregamento do dataset
df = pd.read_excel('../../Data/Raw/projecoes.xlsx', skiprows=5, engine='openpyxl',)

In [3]:
df.head()

,IDADE,SEXO,CÓD.,SIGLA,LOCAL,2000,2001,2002,2003,2004,...,2061,2062,2063,2064,2065,2066,2067,2068,2069,2070
0,0,Ambos,0,BR,Brasil,3423475,3347313,3274356,3212295,3163041,...,1615589,1597609,1580751,1564427,1549026,1534801,1521584,1509151,1497237,1485716
1,1,Ambos,0,BR,Brasil,3450022,3406966,3332612,3261091,3200484,...,1634395,1614666,1596716,1579885,1563579,1548205,1534002,1520805,1508394,1496496
2,2,Ambos,0,BR,Brasil,3461038,3444450,3401900,3327924,3256791,...,1655206,1633932,1614217,1596273,1579457,1563166,1547800,1533609,1520415,1508015
3,3,Ambos,0,BR,Brasil,3469109,3458052,3441638,3399284,3325501,...,1676639,1654738,1633474,1613776,1595841,1579039,1562761,1547399,1533216,1520035
4,4,Ambos,0,BR,Brasil,3477903,3466901,3455987,3439662,3397467,...,1697627,1676166,1654275,1633030,1613338,1595416,1578625,1562356,1547006,1532831


In [4]:
df.columns

Index(['IDADE',  'SEXO',  'CÓD.', 'SIGLA', 'LOCAL',    2000,    2001,    2002,
          2003,    2004,    2005,    2006,    2007,    2008,    2009,    2010,
          2011,    2012,    2013,    2014,    2015,    2016,    2017,    2018,
          2019,    2020,    2021,    2022,    2023,    2024,    2025,    2026,
          2027,    2028,    2029,    2030,    2031,    2032,    2033,    2034,
          2035,    2036,    2037,    2038,    2039,    2040,    2041,    2042,
          2043,    2044,    2045,    2046,    2047,    2048,    2049,    2050,
          2051,    2052,    2053,    2054,    2055,    2056,    2057,    2058,
          2059,    2060,    2061,    2062,    2063,    2064,    2065,    2066,
          2067,    2068,    2069,    2070],
      dtype='object')

In [5]:
def preparar_dados_crescimento(df, local='Brasil'):
    """
    Prepara dados de crescimento populacional filtrados por local
    """
    # Detectar o nome correto da coluna (case-insensitive)
    coluna_local = None
    for col in df.columns:
        if col.upper() == 'LOCAL':
            coluna_local = col
            break
    
    if coluna_local is None:
        raise ValueError("Coluna 'LOCAL' não encontrada no DataFrame")
    
    # Filtrar por local
    df_local = df[df[coluna_local] == local].copy()
    
    if df_local.empty:
        raise ValueError(f"Nenhum dado encontrado para o local '{local}'")
    
    # Identificar colunas de anos (numéricas)
    colunas_anos = [col for col in df_local.columns if isinstance(col, (int, float)) or (isinstance(col, str) and col.isdigit())]
    colunas_anos = sorted([int(col) for col in colunas_anos])
    
    # Criar DataFrame pivotado
    dados_crescimento = []
    
    for _, row in df_local.iterrows():
        sexo = row['SEXO']
        for ano in colunas_anos:
            dados_crescimento.append({
                'Ano': ano,
                'Sexo': sexo,
                'Populacao': row[ano]
            })
    
    df_pivot = pd.DataFrame(dados_crescimento)
    
    # Pivotar para ter colunas por sexo
    df_resultado = df_pivot.pivot_table(
        index='Ano',
        columns='Sexo',
        values='Populacao',
        aggfunc='sum'
    )
    
    # Renomear colunas
    df_resultado.columns.name = None
    
    return df_resultado


In [6]:
def criar_grafico_interativo(df, local='Brasil'):
    df_crescimento = preparar_dados_crescimento(df, local)
    
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.65, 0.35],
        subplot_titles=('', ''),  # Vazio - vamos adicionar manualmente
        vertical_spacing=0.18,
        specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
    )
    
    # Paleta de cores moderna
    cores = {
        'ambos': '#6366f1',
        'homens': '#3b82f6',
        'mulheres': '#ec4899',
        'diferenca': '#10b981'
    }
    
    # ===== GRÁFICO PRINCIPAL =====
    fig.add_trace(go.Scatter(
        x=df_crescimento.index, 
        y=df_crescimento['Ambos'],
        name='Total',
        line=dict(color=cores['ambos'], width=3),
        fill='tozeroy',
        fillcolor='rgba(99, 102, 241, 0.1)',
        hovertemplate='<b>Ano:</b> %{x}<br><b>População Total:</b> %{y:,.0f}<extra></extra>'
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        x=df_crescimento.index, 
        y=df_crescimento['Homens'],
        name='Homens',
        line=dict(color=cores['homens'], width=2.5, dash='dash'),
        hovertemplate='<b>Ano:</b> %{x}<br><b>Homens:</b> %{y:,.0f}<extra></extra>'
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        x=df_crescimento.index, 
        y=df_crescimento['Mulheres'],
        name='Mulheres',
        line=dict(color=cores['mulheres'], width=2.5, dash='dot'),
        hovertemplate='<b>Ano:</b> %{x}<br><b>Mulheres:</b> %{y:,.0f}<extra></extra>'
    ), row=1, col=1)
    
    # Anotações nos extremos
    anos_destaque = [df_crescimento.index.min(), df_crescimento.index.max()]
    for ano in anos_destaque:
        if ano in df_crescimento.index:
            fig.add_annotation(
                x=ano,
                y=df_crescimento.loc[ano, 'Ambos'],
                text=f"<b>{df_crescimento.loc[ano, 'Ambos']:,.0f}</b>",
                showarrow=True,
                arrowhead=2,
                arrowsize=1,
                arrowwidth=2,
                arrowcolor=cores['ambos'],
                ax=0,
                ay=-40,
                font=dict(size=11, color=cores['ambos']),
                bgcolor='rgba(255, 255, 255, 0.8)',
                bordercolor=cores['ambos'],
                borderwidth=1,
                borderpad=4,
                row=1, col=1
            )
    
    # ===== GRÁFICO DE DIFERENÇA =====
    diferenca = df_crescimento['Mulheres'] - df_crescimento['Homens']
    step = max(1, len(diferenca) // 50)
    
    fig.add_trace(go.Bar(
        x=diferenca.index[::step], 
        y=diferenca.values[::step],
        name='Diferença (M - H)',
        marker=dict(
            color=cores['diferenca'],
            line=dict(width=0)
        ),
        hovertemplate='<b>Ano:</b> %{x}<br><b>Diferença:</b> %{y:,.0f}<extra></extra>'
    ), row=2, col=1)
    
    fig.add_hline(y=0, line_dash="dash", line_color="gray", 
                    opacity=0.5, row=2, col=1)
    
    # ===== LAYOUT GERAL =====
    fig.update_layout(
        height=900,
        showlegend=True,
        title={
            'text': f"<b style='font-size:22px'>Análise Populacional Completa - {local}</b><br>" +
                    f"<span style='font-size:14px; color:#666'>Período: {df_crescimento.index.min()} a {df_crescimento.index.max()}</span>",
            'x': 0.5,
            'xanchor': 'center',
            'y': 0.98,
            'yanchor': 'top',
            'pad': {'b': 30}
        },
        hovermode='x unified',
        template='plotly_white',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.06,
            xanchor="right",
            x=1,
            font=dict(size=12),
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='rgba(0, 0, 0, 0.1)',
            borderwidth=1
        ),
        font=dict(family="Arial, sans-serif", size=12),
        margin=dict(t=120, b=80, l=90, r=80)
    )
    
    # ===== ADICIONAR TÍTULOS DOS SUBPLOTS MANUALMENTE =====
    # Título do primeiro gráfico (Projeção) - POSIÇÃO AJUSTADA PARA CIMA
    fig.add_annotation(
        text=f"<b>Projeção Populacional - {local}</b>",
        xref="paper", yref="paper",
        x=0.5, y=0.975,  # ⬆️ AUMENTADO de 0.95 para 0.975
        xanchor='center', yanchor='bottom',
        showarrow=False,
        font=dict(size=15, color='#1f2937', family='Arial'),
        bgcolor='rgba(255, 255, 255, 0.9)',
        borderpad=8
    )
    
    # Título do segundo gráfico (Diferença)
    fig.add_annotation(
        text="<b>Diferença entre Mulheres e Homens</b>",
        xref="paper", yref="paper",
        x=0.5, y=0.395,
        xanchor='center', yanchor='bottom',
        showarrow=False,
        font=dict(size=15, color='#1f2937', family='Arial'),
        bgcolor='rgba(255, 255, 255, 0.9)',
        borderpad=8
    )
    
    # ===== EIXOS =====
    # Eixo X - Gráfico superior
    fig.update_xaxes(
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(128, 128, 128, 0.2)',
        row=1, col=1,
        title_font=dict(size=13),
        tickfont=dict(size=11)
    )
    
    # Eixo X - Gráfico inferior
    fig.update_xaxes(
        title_text="<b>Ano</b>", 
        title_standoff=15,
        row=2, col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(128, 128, 128, 0.2)',
        title_font=dict(size=14),
        tickfont=dict(size=11)
    )
    
    # Eixo Y - Gráfico superior
    fig.update_yaxes(
        title_text="<b>População</b>", 
        title_standoff=15,
        row=1, col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(128, 128, 128, 0.2)',
        tickformat=',',
        separatethousands=True,
        title_font=dict(size=14),
        tickfont=dict(size=11)
    )
    
    # Eixo Y - Gráfico inferior
    fig.update_yaxes(
        title_text="<b>Diferença Populacional</b>", 
        title_standoff=15,
        row=2, col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(128, 128, 128, 0.2)',
        tickformat=',',
        separatethousands=True,
        zeroline=True,
        title_font=dict(size=14),
        tickfont=dict(size=11)
    )
    
    return fig

# Uso
fig_interativo = criar_grafico_interativo(df, 'Brasil')
fig_interativo.show()